# 15.13 如何公平比較 Dense／MoE？


如果MoE多存了四套FFN，卻拿它與只有一套、相同寬度的Dense比較，品質提升可能只是來自更多參數。反過來，若兩者存一樣多，MoE每個token又可能用較少計算。公平比較必須先回答：我們想固定儲存容量，還是固定每個token的大致工作量？前置是[總權重與使用估計](https://birdhackor.github.io/tiny-perceptron-vlm/15.10.html)，以及[實測耗時](https://birdhackor.github.io/tiny-perceptron-vlm/15.11.html)；這兩種固定方式不會自然產生同一個Dense尺寸。

用不含bias的玩具尺寸推算：特徵寬度4，每位expert中間寬度2，單位有 `4×2+2×4=16` 個權重。四位共64，router另有16，總共80。若Dense也要80個權重，它的中間寬度應是10，因為 `4×10+10×4=80`。但MoE若top-1，每token使用估計只有 `16+16=32`；按這個估計配Dense，寬度4就足夠。兩種「匹配」得到10和4，回答的是不同問題。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
width, expert_hidden, experts = 4, 2, 4
per_expert = 2 * width * expert_hidden
router = width * experts
total = experts * per_expert + router
for k in (1, 2):
    active = k * per_expert + router
    same_parameters_hidden = total / (2 * width)
    same_active_estimate_hidden = active / (2 * width)
    print("k", k, "MoE總權重", total, "使用估計", active)
    print("同總參數Dense寬度", same_parameters_hidden, "同使用估計Dense寬度", same_active_estimate_hidden)

程式輸入都是架構尺寸，沒有訓練或計時。k=1時總權重80、使用估計32，兩個Dense寬度10與4；k=2時總權重仍80，使用估計48，寬度10與6。這裡結果剛好是整數，真實尺寸可能需要取整並報出剩餘差距。你應從輸出看見top-k影響工作量，不影響已存的expert套數。

按使用權重數匹配只是計算成本的初步代理，不能直接稱為精確FLOPs匹配。FLOPs是浮點運算次數，還需把router、啟動函數、合併，以及完整模型的attention等納入。即使運算次數接近，索引與搬移成本仍可能讓時間不同，所以應另列實測耗時、最高記憶體占用與[路由負載](https://birdhackor.github.io/tiny-perceptron-vlm/15.8.html)：各expert被選次數，以及次數占總分派筆數的比例。參數匹配則應數實際模組的全部參數，包含bias及共享部分，不能只沿用此簡化算式。

完成尺寸選擇後，兩個模型還應使用相同訓練資料、相同已讀token數、相同[驗證集](https://birdhackor.github.io/tiny-perceptron-vlm/5.10.html)，也就是留作比較方案、其誤差不直接用來更新權重的資料。再重複幾個隨機種子：種子指定亂數起點，讓初始化可重現；換幾個起點能檢查差異是否只是偶然。若只用不同資料訓練各一次，結果混入資料與偶然性，無法歸因架構。最後分別報「同總參數」和「同計算預算」的比較，讓讀者知道結論在什麼資源條件下成立。

練習只把experts由4改成8，k保持1。先手算總權重160、使用估計48，同總參數Dense寬度20、同使用估計寬度6，再執行核對。這也提醒router隨expert數變大，計算匹配時不能假設它始終免費。
